In [1]:
from pyspark.sql import SparkSession 

In [3]:
spark = SparkSession.builder.appName("Case Study").getOrCreate()

In [11]:
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)

In [12]:
total_customers = customers.count()
total_products = products.count()
total_orders = orders.count()
total_returned_orders = returns.count()

In [13]:
print(f"Total Customers: {total_customers}")
print(f"Total Products: {total_products}")
print(f"Total Orders: {total_orders}")
print(f"Total Returned Orders: {total_returned_orders}")

Total Customers: 100000
Total Products: 50000
Total Orders: 1000000
Total Returned Orders: 100000


In [14]:
from pyspark.sql.functions import sum, col
order_items = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
sales_data = order_items.join(products, "product_id")

sales_data.groupBy("category") \
    .agg(
        sum(col("selling_price") * col("quantity"))
        .alias("total_sales")
    ) \
    .show()

[Stage 39:=============================>                            (1 + 1) / 2]

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|Home & Kitchen|7.581388732799902E8|
|        Sports|7.433388681300008E8|
|   Electronics|7.442665041099958E8|
|      Clothing|7.419227945699946E8|
|         Books|7.464907783499908E8|
|        Beauty|7.626693058999963E8|
|          Toys|7.446190722999846E8|
+--------------+-------------------+



In [15]:
from pyspark.sql.functions import sum, col

top_customers = orders.join(order_items, "order_id") \
    .groupBy("customer_id") \
    .agg(
        sum(col("selling_price") * col("quantity"))
        .alias("total_purchase")
    ) \
    .orderBy(col("total_purchase").desc()) \
    .limit(10)

top_customers.show()

[Stage 46:=============================>                            (1 + 1) / 2]

+-----------+------------------+
|customer_id|    total_purchase|
+-----------+------------------+
|      93094|         181569.68|
|      64560|169060.39999999997|
|      23289|161573.80000000002|
|      52275|153364.78999999998|
|      61218|         153067.55|
|      52034|         152680.05|
|      40442|         151037.32|
|      60528|         148691.95|
|      84830|         148363.84|
|      82593|148281.03999999998|
+-----------+------------------+

